### 1. Обзор структуры ocr_dataset.csv

В этом шаге загружается файл `data/ocr_rvl_cdip/ocr_dataset.csv`, чтобы понять его структуру: список колонок, размер датасета и несколько первых строк. Это нужно для дальнейшего проектирования текстового ML-пайплайна после OCR.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent
csv_path = project_root / "data" / "ocr_rvl_cdip" / "ocr_dataset.csv"

print(f"CSV path: {csv_path}")
df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))

print("\nHead:")
display(df.head())

print("\nLabel value counts (top 20):")
display(df["label"].value_counts().head(20))

CSV path: /home/dovzhuk/projects/document-checker-course/data/ocr_rvl_cdip/ocr_dataset.csv
Shape: (2544, 4)

Columns: ['path', 'label', 'reference', 'text']

Head:


,path,label,reference,text
0,images_tif/advertisement/0000001863.tif,advertisement,advertisement/0000001863,IMAGE\nNOT\nAVAILABLE\nONLINE\n\nThe material ...
1,images_tif/advertisement/00001014.tif,advertisement,advertisement/00001014,0009\n\nARCHIVE LOCATION SHEET\n\nTHE NUMBER (...
2,images_tif/advertisement/0000106842.tif,advertisement,advertisement/0000106842,20 CLASS A CIGARETTES\nSIERRA\nMENTHOL\n\nIntr...
3,images_tif/advertisement/0000107404.tif,advertisement,advertisement/0000107404,PRS Eye Movement\nTracking\n660104374\nPERCEPT...
4,images_tif/advertisement/0000109229.tif,advertisement,advertisement/0000109229,"Several weeks ago, we left you samples of Arct..."



Label value counts (top 20):


label
file_folder               383
email                     176
memo                      173
budget                    170
scientific_report         170
advertisement             167
presentation              163
invoice                   158
form                      157
questionnaire             155
handwritten               148
specification             126
letter                    115
resume                    110
news_article               87
scientific_publication     86
Name: count, dtype: int64

### Вывод по пункту 1

Файл `data/ocr_rvl_cdip/ocr_dataset.csv` содержит 2544 строки и 4 колонки: `path`, `label`, `reference`, `text`. Колонка `text` хранит результат OCR по каждому изображению из `images_tif`, а колонка `label` задаёт тип документа (16 классов, таких как `file_folder`, `email`, `memo`, `budget`, `advertisement` и т.д.). Баланс классов относительно ровный: самый частый класс (`file_folder`) содержит около 380 примеров, самый редкий (`scientific_publication`) около 80, что позволяет начать с простых линейных моделей без сложной балансировки.

### 2. Разделение на train/val и базовая фильтрация

На этом шаге отфильтруем строки с пустым или слишком коротким текстом и разделим датасет на обучающую и валидационную части. Это подготовит данные для обучения базовой текстовой модели классификации типа документа.

In [2]:
from sklearn.model_selection import train_test_split

# Копия исходного датафрейма, чтобы не портить df
df_clean = df.copy()

# Убираем строки с пустым или NaN-текстом
df_clean["text"] = df_clean["text"].astype(str).str.strip()
before = len(df_clean)
df_clean = df_clean[df_clean["text"].str.len() > 0]
after = len(df_clean)

print(f"Removed empty texts: {before - after} rows")

# Приводим к обычным Python-спискам
X = df_clean["text"].astype(str).tolist()
y = df_clean["label"].astype(str).tolist()

# Разделение на train / val
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}")

Removed empty texts: 3 rows
Train size: 2032, Val size: 509


### Вывод по пункту 2

После фильтрации пустых OCR-текстов из датасета было удалено всего 3 строки, что не влияет существенно на объём данных. Оставшиеся примеры (2541 строка) были разделены на обучающую и валидационную части с долей 80/20 и стратификацией по метке `label`: 2032 документа в train и 509 в val. Это обеспечивает репрезентативное покрытие всех 16 классов как на обучении, так и на валидации.

### 3. Базовая модель: TF‑IDF + Logistic Regression

На этом шаге строим базовый текстовый пайплайн: TF‑IDF-векторизатор по полю `text` и линейный классификатор (Logistic Regression). Модель обучим на `X_train, y_train` и оценим на `X_val, y_val` по метрикам accuracy и macro F1.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Базовый TF-IDF + Logistic Regression пайплайн
text_clf = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(
            max_features=50000,
            ngram_range=(1, 2),
            lowercase=True,
        )),
        ("clf", LogisticRegression(
            max_iter=1000,
            n_jobs=-1,
            class_weight=None,
        )),
    ]
)

# Обучение
text_clf.fit(X_train, y_train)

# Предсказание на валидации
y_val_pred = text_clf.predict(X_val)

acc = accuracy_score(y_val, y_val_pred)
f1_macro = f1_score(y_val, y_val_pred, average="macro")

print(f"Validation accuracy: {acc:.4f}")
print(f"Validation macro F1: {f1_macro:.4f}")

print("\nClassification report:")
print(classification_report(y_val, y_val_pred))

/home/dovzhuk/projects/document-checker-course/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Validation accuracy: 0.6857
Validation macro F1: 0.6763

Classification report:
                        precision    recall  f1-score   support

         advertisement       0.79      0.65      0.71        34
                budget       0.62      0.44      0.52        34
                 email       0.97      0.91      0.94        35
           file_folder       0.53      1.00      0.69        76
                  form       0.60      0.48      0.54        31
           handwritten       0.71      0.33      0.45        30
               invoice       0.88      0.88      0.88        32
                letter       0.77      0.74      0.76        23
                  memo       0.64      0.80      0.71        35
          news_article       0.65      0.65      0.65        17
          presentation       0.75      0.36      0.49        33
         questionnaire       0.92      0.74      0.82        31
                resume       0.89      0.77      0.83        22
scientific_publication 

### Вывод по пункту 3

Базовый текстовый пайплайн TF‑IDF + Logistic Regression дал на валидации качество около 0.69 по accuracy и 0.68 по macro F1. Для 16 классов документов и неидеальных OCR-текстов это неплохой стартовый бейзлайн. Модель хорошо распознаёт классы с более стабильной текстовой структурой (`email`, `invoice`, `questionnaire`, `resume`), но заметно хуже справляется с более шумными и разнообразными по содержанию классами (`handwritten`, `presentation`, частично `budget` и `scientific_report`). Эти результаты можно использовать как отправную точку для последующих улучшений (очистка текста, настройка TF‑IDF и гиперпараметров, альтернативные линейные модели).

### 4. Сохранение обученного пайплайна

На этом шаге сохраняем обученный пайплайн `TF‑IDF + Logistic Regression` в каталог `artifacts/models/` с помощью `joblib`. Это позволит в дальнейшем загружать модель без повторного обучения и использовать её в основном приложении и на демо.

In [4]:
from pathlib import Path
import joblib

project_root = Path.cwd().parent
models_dir = project_root / "artifacts" / "models"
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "text_tfidf_logreg.joblib"

joblib.dump(text_clf, model_path)

print(f"Model saved to: {model_path}")

Model saved to: /home/dovzhuk/projects/document-checker-course/artifacts/models/text_tfidf_logreg.joblib


### Вывод по пункту 4

Обученный пайплайн `TF‑IDF + Logistic Regression` сохранён в виде артефакта по пути `artifacts/models/text_tfidf_logreg.joblib`. Теперь модель можно загружать без повторного обучения и использовать в основном коде и демо-сценариях (`document → OCR → text → ML → result`). Это закрывает базовое требование курсового проекта по наличию сохранённой воспроизводимой текстовой модели.

### 5. Лёгкая очистка текста и модель Linear SVM

На этом шаге добавляем простую предобработку OCR-текста (нормализация регистра и пробелов) и обучаем альтернативную модель на тех же данных: TF‑IDF + LinearSVC (Linear SVM). Цель — проверить, удастся ли улучшить качество по сравнению с бейзлайном TF‑IDF + Logistic Regression.

In [5]:
import re
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

class SimpleTextCleaner(BaseEstimator, TransformerMixin):
    """Простая очистка текста: lower, убираем лишние пробелы и управляющие символы."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        cleaned = []
        for text in X:
            t = str(text)
            t = t.lower()
            # Убираем управляющие символы и лишние пробелы
            t = re.sub(r"\s+", " ", t)
            t = t.strip()
            cleaned.append(t)
        return cleaned

# Пайплайн: очистка текста -> TF-IDF -> Linear SVM
svm_clf = Pipeline(
    steps=[
        ("clean", SimpleTextCleaner()),
        ("tfidf", TfidfVectorizer(
            max_features=50000,
            ngram_range=(1, 2),
            lowercase=False,  # lowercase уже делаем в cleaner
        )),
        ("clf", LinearSVC())
    ]
)

# Обучение
svm_clf.fit(X_train, y_train)

# Предсказание на валидации
y_val_pred_svm = svm_clf.predict(X_val)

acc_svm = accuracy_score(y_val, y_val_pred_svm)
f1_macro_svm = f1_score(y_val, y_val_pred_svm, average="macro")

print(f"Validation accuracy (LinearSVC): {acc_svm:.4f}")
print(f"Validation macro F1 (LinearSVC): {f1_macro_svm:.4f}")

print("\nClassification report (LinearSVC):")
print(classification_report(y_val, y_val_pred_svm))

Validation accuracy (LinearSVC): 0.7623
Validation macro F1 (LinearSVC): 0.7429

Classification report (LinearSVC):
                        precision    recall  f1-score   support

         advertisement       0.87      0.76      0.81        34
                budget       0.66      0.68      0.67        34
                 email       0.94      0.97      0.96        35
           file_folder       0.74      0.99      0.85        76
                  form       0.63      0.55      0.59        31
           handwritten       0.88      0.47      0.61        30
               invoice       0.88      0.88      0.88        32
                letter       0.72      0.91      0.81        23
                  memo       0.79      0.74      0.76        35
          news_article       0.63      0.71      0.67        17
          presentation       0.68      0.39      0.50        33
         questionnaire       0.86      0.81      0.83        31
                resume       0.83      0.86      0.

### Вывод по пункту 5

Модель TF‑IDF + LinearSVC с простой очисткой текста заметно улучшила качество по сравнению с бейзлайном Logistic Regression. Валидационная accuracy выросла примерно с 0.69 до 0.76, а macro F1 — с ~0.68 до ~0.74. Больше всего выиграли классы с более стабильной структурой текста (`budget`, `scientific_report`, `file_folder`), но даже для сложных и шумных классов (`handwritten`, `presentation`, `scientific_publication`) качество стало лучше. На текущем этапе эту модель логично считать основной кандидатной текстовой моделью для интеграции в общий пайплайн.

### 6. Сохранение улучшенной модели LinearSVC

После того как модель TF‑IDF + LinearSVC показала лучшее качество на валидации, на этом шаге сохраняем её в каталог `artifacts/models/` как отдельный артефакт. Это позволит использовать LinearSVC в основном приложении и при демонстрации без повторного обучения, а также сравнивать её с базовой моделью Logistic Regression.

In [6]:
from pathlib import Path
import joblib

project_root = Path.cwd().parent
models_dir = project_root / "artifacts" / "models"
models_dir.mkdir(parents=True, exist_ok=True)

svm_model_path = models_dir / "text_tfidf_linearsvc.joblib"

joblib.dump(svm_clf, svm_model_path)

print(f"SVM model saved to: {svm_model_path}")

SVM model saved to: /home/dovzhuk/projects/document-checker-course/artifacts/models/text_tfidf_linearsvc.joblib


### Вывод по шагу 6

Улучшенный пайплайн `SimpleTextCleaner → TF‑IDF → LinearSVC` сохранён как артефакт по пути `artifacts/models/text_tfidf_linearsvc.joblib`. Теперь в проекте есть два варианта текстовой модели: базовый (Logistic Regression) и улучшенный (LinearSVC). Для демонстрации и интеграции в общий пайплайн разумно использовать именно LinearSVC как текущую лучшую модель по валидационным метрикам.